# Imports & Setup

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch import nn
import matplotlib.pyplot as plt
import os
import numpy as np
import imageio
from collections import OrderedDict
from typing import Dict, Callable
from IPython.display import display, Markdown

In [ ]:
from transformers import LogitsProcessor, LogitsProcessorList

class GenerationStep:
    def __init__(self, logits: torch.FloatTensor, preceding_token: str):
        self.logits = logits
        self.preceding_token = preceding_token


class CustomLogitsProcessor(LogitsProcessor):
    def __init__(self, scores):
        self.scores = scores

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        step = GenerationStep(logits=scores, preceding_token=input_ids[0][-1].item())
        

class Model:
    def __init__(self, model_name: str, device: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype='auto',
            device_map=device
        )
        self.device = device
    
    def __repr__(self):
        return \
            f"------------------- Model -------------------\n{self.model.__class__.__name__}\n" + \
            f"----------------- Tokenizer -----------------\n{self.tokenizer.__class__.__name__}\n" + \
            f"------------------- Device ------------------\n{self.device}\n"
    
    def prepare_inputs(self, prompt: str):
        prompt = "Solve this equation for x: 2x + 5 = 17. Please reason step by step, and put your final answer within \boxed{}."
        messages = [
            {"role": "user", "content": prompt}
        ]
        tokenized_chat = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True
        )
        inputs = self.tokenizer([tokenized_chat], return_tensors="pt").to(self.device)

        return inputs
    
    def generate(
            self,
            inputs,
            max_new_tokens: int = 256,
            temperature: float = 0.0,
            top_p: float = 0.95,
            top_k: int = 20,
            custom_logits_processors: list = None
    ):
        with torch.inference_mode():
            generation_output = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                output_scores=True,
                logits_processor=LogitsProcessorList(custom_logits_processors)
            )
        
        scores = generation_output.scores
        output_ids = generation_output.sequences[0][len(inputs.input_ids[0]):].tolist()
        response = self.tokenizer.decode(output_ids, skip_special_tokens=True)

        return (response, scores)


In [ ]:
model_name = "Qwen/Qwen3-4B"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = Model(model_name, device)
model

'cpu'